# BT4012 Kaggle competition: final model

This notebook rebuilds my final submissions from the four competition CSV files:

- **Primary:** `(2/3)·B + (1/3)·T`, public AUC 0.95358
- **Second:** `0.5·B + 0.5·T`, public AUC 0.95340

**B** is a staged ensemble: recency-weighted LightGBM → three-class annotation-status model → availability-matched graph LightGBM → directed message-passing network (public 0.95261 on its own). **T** is a five-seed TabM trained on the raw 165 features (public 0.95034 on its own).

Sections 1–5 build B (CPU). Section 6 trains TabM (needs a GPU, e.g. a Kaggle notebook with a T4). Section 7 blends and writes both CSVs. Section 8 is optional temporal validation.

Unknown labels are never treated as licit, and only the supplied competition data is used. Put the data in `data/` or set `BT4012_DATA`. Exact probabilities can differ slightly across hardware/library versions.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
DATA = Path(os.environ.get('BT4012_DATA', str(ROOT / 'data'))).expanduser().resolve()
RUN = ROOT / 'runs'
RUN.mkdir(exist_ok=True)
for filename in ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']:
    assert (DATA / filename).is_file(), f'Missing {filename}: set BT4012_DATA or copy supplied files into data/'
def run_script(path, *arguments):
    subprocess.run([sys.executable, str(ROOT / path), *map(str, arguments)], check=True, cwd=ROOT)


## 1. Audit data and separate known from unknown labels

The supplied features have no null values. The label column does. IDs are used for alignment, and time is used for weights and validation.

In [ ]:
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
sample = pd.read_csv(DATA / 'sample_submission.csv')
features = [f'feat_{i}' for i in range(1, 166)]
assert not train[features].isna().any().any()
assert not test[features].isna().any().any()
assert train.txId.is_unique and test.txId.is_unique
assert set(train.txId).isdisjoint(test.txId)
assert train.time_step.max() < test.time_step.min()
assert set(train.label.dropna().unique()) <= {0, 1}
assert np.array_equal(test['index'], sample['index'])
print({'train_rows': len(train), 'known': int(train.label.notna().sum()), 'unknown': int(train.label.isna().sum()), 'test_rows': len(test)})
print(train.label.value_counts(dropna=False))


## 2. Reference and annotation-status model

Fit five seeds with float32 inputs and weights exp((period-35)/5). The auxiliary model predicts licit, illicit or annotation-unknown, then conditions on the two known classes. Combine 80/20 in log-odds space. A supplied original recency CSV, if present, is checked for numerical reproduction; it is optional.

In [ ]:
run_script('src/recency/finalize.py', '--data', DATA, '--cache', RUN/'raw_cache', '--out', RUN/'annotation', '--lock', ROOT/'src/recency/selected_lock.json')


## 3. Graph preprocessing and matched neighbour visibility

Retain all graph nodes, hide unknown training vectors for fraud graph inputs, and retain observed known/query vectors. Six relations give 990 means, plus availability and topology. The original 165 query features remain. Cache preparation runs separately from Torch to avoid numerical-runtime deadlocks observed on macOS.

In [ ]:
run_script('src/messages/prepare_caches.py', '--data', DATA, '--graph-cache', RUN/'graph_cache', '--node-cache', RUN/'node_cache')
run_script('src/graph/finalize_graph.py', '--data', DATA, '--cache', RUN/'graph_cache', '--anchor', RUN/'annotation/submission_unlabelled_blend.csv', '--out', RUN/'graph')


## 4. Directed message network and final prediction

Two residual layers encode directed neighbourhoods and a learned missing-node state. Pretraining reconstructs supplied past features; the fraud classifier uses only known training labels. A past-only inner 32/33-35 window selects the epoch count. The original final fit selected 16 supervised epochs. Five neural seeds are averaged, then blended 20% with 80% of the masked graph ensemble in log-odds space. This is the selected formula, not a new coefficient search.

In [ ]:
run_script('src/messages/learned_messages/finalize_probe.py', '--data', DATA, '--node-cache', RUN/'node_cache', '--graph-cache', RUN/'graph_cache', '--anchor', RUN/'graph/submission_missing_neighbours_conservative.csv', '--lock', ROOT/'src/messages/learned_messages/neural_candidate_lock.json', '--out', RUN/'final')


## 5. Check ensemble B

Each finalizer saves and reloads its models and checks the predictions. B must follow the sample index exactly.

In [ ]:
submission = pd.read_csv(RUN/'final/submission_neural_graph_probe.csv')
assert submission.columns.tolist() == ['index', 'target']
assert len(submission) == len(sample) == 15329
assert submission['index'].is_unique
np.testing.assert_array_equal(submission['index'], sample['index'])
assert np.isfinite(submission.target).all() and submission.target.between(0, 1).all()
checks = json.loads((RUN/'final/submission_checks.json').read_text())
assert checks['all_models_and_transformer_reloaded'] and checks['formula_exactly_reproduced']
print(checks)
submission.head()


## 6. TabM (T)

TabM with 32 implicit ensemble members, three 512-wide blocks, dropout 0.1 and piecewise-linear embeddings (48 bins, 16 dims). Inputs are quantile-normalised using statistics from the training rows only. Uniform sample weights (no recency weighting).

The number of epochs is chosen on an inner window (train ≤32, validate 33–35), which picked 17. The model is then refit on all labelled rows from steps 1–35 with seeds 0–4. Each seed saves its weights, reloads them and checks the predictions match.

In [ ]:
import torch
TABM_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'   # CPU works but is slow
run_script('src/tabm/train_tabm.py', '--data', DATA, '--root', ROOT, '--out', RUN/'tabm',
           '--models', 'tabm_ple_uniform', '--final', 'tabm_ple_uniform', '--seeds', '0,1,2,3,4', '--device', TABM_DEVICE)

## 7. Blend B and T and write the submissions

TabM predictions are matched to the submission index through `txId`, averaged over the five seeds, then blended with B in probability space.

In [ ]:
seed_dirs = sorted((RUN/'tabm/tabm_ple_uniform/fold_35_49').glob('seed_*'))
assert len(seed_dirs) == 5, seed_dirs
tabm_seeds = [pd.read_csv(d/'predictions.csv').set_index('txId')['p'] for d in seed_dirs]
T = pd.concat(tabm_seeds, axis=1).mean(axis=1).reindex(test.txId).to_numpy()
B = pd.read_csv(RUN/'final/submission_neural_graph_probe.csv').set_index('index').loc[test['index'], 'target'].to_numpy()
assert np.isfinite(T).all() and np.isfinite(B).all()

OUT = ROOT/'submissions_rebuilt'
OUT.mkdir(exist_ok=True)
for name, p in {'submission_tabm_blend33.csv': (2/3)*B + (1/3)*T,
                'submission_tabm_ple_uniform_blend50.csv': 0.5*B + 0.5*T}.items():
    sub = pd.DataFrame({'index': test['index'], 'target': p})
    assert len(sub) == 15329 and sub['index'].is_unique and sub.target.between(0, 1).all()
    np.testing.assert_array_equal(sub['index'], sample['index'])
    sub.to_csv(OUT/name, index=False)
    print(name, sub.target.describe()[['mean', 'min', 'max']].round(4).to_dict())

In [ ]:
# Optional: compare with the submitted files in submissions/ (small differences are expected on different hardware)
for name in ['submission_tabm_blend33.csv', 'submission_tabm_ple_uniform_blend50.csv']:
    old = pd.read_csv(ROOT/'submissions'/name).target
    new = pd.read_csv(OUT/name).target
    print(name, 'max abs diff', float(np.abs(old - new).max()), '| rank corr', round(old.rank().corr(new.rank()), 6))

## 8. Optional chronological validation

This is expensive. It refits three seeds at cutoffs 25, 28 and 30, then evaluates 26-30, 29-32 and 31-35. Models and transformations use only past prefixes. AUC is calculated after seed-probability averaging, then averaged equally over the three windows. A ten-period and fourteen-period stress window are also available. Saved report values are historical evidence, not outputs fabricated by this notebook. Keep evaluation output directories separate from final inference.

In [ ]:
RUN_VALIDATION = False
if RUN_VALIDATION:
    run_script('src/messages/period_relative/period_experiment.py', '--data', DATA, '--graph-cache', RUN/'graph_cache', '--old-root', RUN/'empty_prior', '--out', RUN/'validation_controls', '--cutoffs', '25:35,28:32,30:35,21:35,32:35', '--controls-only')
    run_script('src/messages/learned_messages/message_model.py', '--node-cache', RUN/'node_cache', '--graph-cache', RUN/'graph_cache', '--controls', RUN/'validation_controls', '--out', RUN/'validation_neural', '--models', 'pretrained', '--folds', '25:35,28:32,30:35,21:35,32:35')
    display(pd.read_csv(RUN/'validation_neural/results.csv'))
